## 연습 1: TelConnect Unity Catalog 구조 설정

이 연습에서는 Lakeflow Job 작업이 읽고 쓸 Unity Catalog 기반을 준비합니다. 메달리온 아키텍처 패턴을 따르는 3개의 스키마가 있는 카탈로그를 생성합니다:

- `bronze` — 수집된 원본 CDR 레코드
- `silver` — 정리되고 검증된 CDR 데이터
- `gold` — 보고용 집계된 네트워크 사용 지표

### ✅ 제공됨: 처리 날짜에 대한 작업 매개변수 수락

이 노트북이 Lakeflow Job 작업으로 실행될 때, 작업은 `processing_date` 매개변수를 자동으로 전달합니다. `dbutils.widgets` API는 노트북 내에서 해당 값을 사용 가능하게 합니다.

아래 셀을 실행하여 위젯을 등록하십시오. 수동으로 실행할 때는 기본값이 오늘 날짜로 설정됩니다.

In [ ]:
# ✅ Run this cell — registers the job parameter widget

import datetime

dbutils.widgets.text("processing_date", str(datetime.date.today()), "Processing Date (YYYY-MM-DD)")
processing_date = dbutils.widgets.get("processing_date")

print(f"Processing date: {processing_date}")

### ✅ 제공됨: Unity Catalog 구조 생성

아래 셀을 실행하여 `telconnect_lab` 카탈로그 및 3개의 메달리온 스키마를 생성합니다. 이를 실습 전체에서 사용하게 됩니다.


In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")